Designing and implementing an LLM-powered chatbot and this chatbot will remember our previous interaction.

In [1]:
!pip install langchain_groq langchain_community

In [2]:
from posix import environ
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key=os.getenv('GROQ_API_KEY')
os.environ['Langchain_Tracing_v2']="True" #for tracing
os.environ['Langchain_Project']=os.getenv('LANGCHAIN_PROJECT')
os.environ['Langchain_api_key']=os.getenv('LANGCHAIN_API_KEY')

In [3]:
from langchain_groq import ChatGroq

model=ChatGroq(model='groq/compound-mini',groq_api_key=groq_api_key)
model

ChatGroq(profile={}, client=<groq.resources.chat.completions.Completions object at 0x7bf47f09eab0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x7bf47f061e20>, model_name='groq/compound-mini', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [4]:
#conversing with our Model
from langchain_core.messages import HumanMessage

model.invoke([HumanMessage(content='Hey ,This is masha and today i had a bad day csz my phone is not working ')])
# print(response)

AIMessage(content='Hey Masha, I’m really sorry to hear you’re having a rough day—especially when your phone isn’t cooperating. That can be super frustrating!  \n\nIf you’d like, I can walk you through some quick troubleshooting steps to see if we can get it working again:\n\n1. **Restart the phone** – Hold the power button (and volume down, if it’s an Android) for about 10\u202fseconds until it powers off, then turn it back on.  \n2. **Check the battery** – Make sure it’s charged. If the battery is removable, take it out for a few seconds and reseat it.  \n3. **Inspect the charger and cable** – Try a different charger or cable, and see if the phone shows any charging indicator.  \n4. **Look for physical damage** – Any cracks, water exposure, or loose ports can cause issues.  \n5. **Safe mode (Android)** – Boot into safe mode to see if a third‑party app is causing the problem.  \n6. **Force‑restart (iPhone)** – For most iPhones, quickly press and release the volume‑up button, then the v

In [5]:
#validating if it remember our interaction or not
from langchain_core.messages import AIMessage

model.invoke(
    [
        HumanMessage(content='Hey ,My name is masha and today i had a bad day csz my phone is not working '),
        AIMessage(content='Hey Masha,I am really sorry to hear you are having a rough day—especially with a phone that’s not working.'),
        HumanMessage(content='hey , i forget why i was having a bad ..?')
    ]
)

AIMessage(content='I’m sorry you’re feeling that way, Masha.  \nYou mentioned earlier that your phone isn’t working, and that was the main thing that made the day tough.  \n\nIf you’d like, we can try to figure out what’s wrong with the phone—whether it’s a battery issue, a frozen screen, connectivity problems, or something else. Just let me know what’s happening (any error messages, what you see on the screen, etc.), and I’ll do my best to help. If you’d rather talk about something else to take your mind off it, that’s fine too—just tell me what you need. 🌼', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 222, 'prompt_tokens': 573, 'total_tokens': 795, 'completion_time': 0.493726, 'completion_tokens_details': None, 'prompt_time': 0.025914, 'prompt_tokens_details': None, 'queue_time': 0.050549, 'total_time': 0.51964}, 'model_name': 'groq/compound-mini', 'system_fingerprint': None, 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'mo

In [6]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store={}

def get_session_history(session_id:str)->BaseChatMessageHistory:
  if session_id not in store:
    store[session_id]=ChatMessageHistory()
  return store[session_id]


with_message_history=RunnableWithMessageHistory(model,get_session_history)

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3553: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


RunnableWithMessageHistory Actually wraps your model like:

session_id

          → fetch history

          → append old messages

          → send to model

In [7]:
config={'configurable':{'session_id':'chat1'}}

In [8]:
response=with_message_history.invoke(
    [HumanMessage(content='Hey ,My name is masha and today i had a bad day csz my phone is not working ')],
    config=config
)

response.content

'Hey Masha, I’m really sorry to hear you’re having a rough day—dealing with a phone that won’t work is the worst!  \n\nIf you’d like, I can walk you through some basic troubleshooting steps. Just let me know the make and model of your phone (e.g., iPhone\u202f14, Samsung\u202fGalaxy\u202fS23, etc.) and what’s happening (won’t turn on, screen stays black, won’t charge, etc.), and we’ll try to get it sorted out together.  \n\nIn the meantime, take a deep breath and maybe step away from the phone for a few minutes—sometimes a short break helps clear the frustration. I’m here to help when you’re ready!'

In [9]:
with_message_history.invoke(
    [HumanMessage(content="what's my name?")],
    config=config
)

AIMessage(content='You introduced yourself as **Masha**.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 48, 'prompt_tokens': 801, 'total_tokens': 849, 'completion_time': 0.105858, 'completion_tokens_details': None, 'prompt_time': 0.040583, 'prompt_tokens_details': None, 'queue_time': 0.123817, 'total_time': 0.146441}, 'model_name': 'groq/compound-mini', 'system_fingerprint': None, 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e06ff-0985-7091-b555-65f2ed59e391-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 801, 'output_tokens': 48, 'total_tokens': 849})

In [12]:
# changing the session_id
config1={'configurable':{'session_id':'chat2'}}
response=with_message_history.invoke(
    [HumanMessage(content="what's my name ")],
    config=config1

)
response.content

'I’m sorry, but I don’t have any information about your name. If you’d like to share it, feel free to let me know!'